# LeetCode 4005 — Minimum Operations to Make Array Equal III bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-array-equal-iii/
- Độ khó: **Hard**
- Chủ đề: Toán / Lý thuyết số (ước, bội), Đếm tần suất, Phân tích thừa số

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Skeleton của bài Premium không có sẵn; tên phương thức dưới đây là **dự đoán** (`minOperations`), cần đối chiếu lại trên LeetCode.

Skeleton (dự đoán, Python3):

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums`. Mỗi phép biến đổi chọn một phần tử rồi **nhân** nó với một số nguyên `k >= 2`, hoặc **chia** nó cho một ước `k` của nó với `2 <= k < nums[i]`. Hỏi số phép **ít nhất** để mọi phần tử bằng nhau.

Nhìn qua có vẻ phải thử rất nhiều giá trị đích (tới `10^9`) và rất nhiều đường biến đổi. Điều đẹp của bài này: **mỗi phần tử chỉ cần 0, 1 hoặc 2 phép** để tới bất kỳ đích nào, và đích tốt nhất luôn có thể chọn trong số các phần tử có sẵn (hoặc "đích rất lớn" tốn đúng `n` phép). Sau đó, bài trở thành đếm **ước và bội** giữa các phần tử.

Sau notebook này bạn sẽ:

1. Chứng minh **công thức chi phí** từng cặp `cost(a, t) ∈ {0, 1, 2}` (và trường hợp đặc biệt đích bằng `1`).
2. Chứng minh vì sao chỉ cần xét đích là một phần tử của mảng, cộng thêm phương án "đích bội chung lớn" tốn đúng `n` phép.
3. Viết brute force bằng **BFS** trên đồ thị số (có giới hạn) để kiểm chứng.
4. Viết lời giải `O(m²)`, lời giải **sàng bội** cho giá trị nhỏ, và lời giải **phân tích thừa số + liệt kê ước** cho giá trị tới `10^9`.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một phép biến đổi trên một phần tử `x`:

- **Nhân**: `x → x * k`, với `k` là số nguyên `>= 2` (tuỳ ý, không giới hạn trên).
- **Chia**: `x → x / k`, với `2 <= k < x` và `k` chia hết `x`.

Trả về số phép ít nhất để tất cả phần tử bằng nhau (giá trị chung tuỳ ý).

**Ví dụ 1:** `nums = [6,12,8]` → `3`. Đưa tất cả về 6: `12 / 2 = 6`; `8 / 4 = 2` rồi `2 * 3 = 6`.

**Ví dụ 2:** `nums = [5,15,20]` → `2`. Đưa về 5: `15 / 3`, `20 / 4`.

**Ví dụ 3:** `nums = [7,7,7]` → `0`.

**Ràng buộc**

- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`

**Hai chi tiết nhỏ nhưng quan trọng:**

- Điều kiện `k < x` nghĩa là **không được chia ra `1`** (vì `x / k > 1`). Vậy từ một số lớn hơn `1` ta không bao giờ về được `1`.
- Số nguyên tố `p` không có `k` nào thoả `2 <= k < p` chia hết `p`, nên **không chia được**, chỉ nhân được.

In [1]:
from typing import List

EXAMPLES = [
    ([6, 12, 8], 3),
    ([5, 15, 20], 2),
    ([7, 7, 7], 0),
]
for nums, expected in EXAMPLES:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[6, 12, 8] -> kỳ vọng 3
nums=[5, 15, 20] -> kỳ vọng 2
nums=[7, 7, 7] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Ước, bội và toán tử `%`

`a` **chia hết** `b` (`a | b`) nghĩa là `b % a == 0`. Khi đó `a` là **ước** của `b`, còn `b` là **bội** của `a`.

In [2]:
print(12 % 4, 12 % 5)            # 0 2
print(12 % 4 == 0)               # True : 4 là ước của 12
print([d for d in range(1, 13) if 12 % d == 0])   # tất cả ước của 12

0 2
True
[1, 2, 3, 4, 6, 12]


### 3.2. `math.gcd`, `math.lcm` và `math.isqrt`

- `gcd(a, b)`: ước chung lớn nhất. `lcm(a, b)`: bội chung nhỏ nhất.
- `isqrt(n)`: căn bậc hai nguyên (sàn). Dùng để biết khi nào dừng thử chia khi phân tích thừa số: nếu `p * p > x` thì phần còn lại của `x` là số nguyên tố.

In [3]:
from math import gcd, lcm, isqrt
print(gcd(12, 18), lcm(12, 18))       # 6 36
print(isqrt(99), isqrt(100))          # 9 10
print(lcm(*[2, 3, 5, 7]))             # 210 (lcm nhiều số: lcm(*list))

6 36
9 10
210


### 3.3. `Counter`, `defaultdict` và hiểu "chi phí" trên đồ thị số

`Counter` đếm số lần, `defaultdict(int)` cộng dồn. Ta cũng sẽ dùng **BFS** (tìm đường ngắn nhất trên đồ thị không trọng số): đồ thị có đỉnh là các số nguyên, cạnh là các phép biến đổi. Khoảng cách BFS chính là số phép ít nhất.

In [4]:
from collections import Counter, defaultdict, deque
c = Counter([6, 12, 8, 6])
print(c, c[6], c[99])
t = defaultdict(int)
t[3] += 2
t[3] += 5
print(dict(t))

# BFS mini: từ 1, mỗi bước nhân 2 hoặc cộng 1, tới 10 mất mấy bước?
dist = {1: 0}
q = deque([1])
while q:
    x = q.popleft()
    for y in (x * 2, x + 1):
        if y <= 10 and y not in dist:
            dist[y] = dist[x] + 1
            q.append(y)
print(dist[10])

Counter({6: 2, 12: 1, 8: 1}) 2 0
{3: 7}
4


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 `[6, 12, 8]` với đích `6`:

- `6 → 6`: 0 phép.
- `12 → 6`: 1 phép (chia 2).
- `8 → 6`: `8` không chia hết `6`, `6` không chia hết `8`. Làm sao? Chia `8` còn `2` (ước chung), nhân `2` lên `6`: **2 phép**.

Tổng 3. Với đích `24`: `6 → 24` (nhân 4), `12 → 24` (nhân 2), `8 → 24` (nhân 3): 3 phép, cũng 3.

Từ đó ta đoán: **một phần tử `a` tới đích `t` tốn tối đa 2 phép**, và tốn 1 phép nếu một trong hai chia hết cái kia. Thử "đường vòng": `a → a·t → t`. Nhân `a` với `t` (1 phép), rồi chia cho `a` (hợp lệ nếu `a ≥ 2`, vì `2 <= a < a·t` và `a | a·t`). Vậy **luôn ≤ 2 phép** khi `a, t ≥ 2`.

Hãy viết hàm chi phí theo phân tích này và kiểm chứng ở mục 5 bằng BFS thật sự.

In [5]:
INF = float("inf")

def pair_cost(a, t):
    """Số phép tối thiểu để đưa a về t (công thức đoán, sẽ kiểm chứng bằng BFS)."""
    if a == t:
        return 0
    if t == 1:
        return INF                    # chỉ số 1 mới "về" được 1
    if a % t == 0 or t % a == 0:
        return 1                      # chia một lần hoặc nhân một lần
    return 2                          # đường vòng a -> a*t -> t

for a, t in [(6, 6), (12, 6), (6, 12), (8, 6), (2, 3), (1, 7), (7, 1), (1, 1)]:
    print(f"cost({a}, {t}) = {pair_cost(a, t)}")

cost(6, 6) = 0
cost(12, 6) = 1
cost(6, 12) = 1
cost(8, 6) = 2
cost(2, 3) = 2
cost(1, 7) = 1
cost(7, 1) = inf
cost(1, 1) = 0


## 5. Cách 1 — Brute Force: BFS trên đồ thị số có giới hạn

**Ý tưởng.** Đây là bản dịch trực tiếp của đề:

1. Giới hạn vũ trụ số trong `[1, CAP]` (ví dụ `CAP = 8000` khi giá trị ban đầu `≤ 8`).
2. Đỉnh = số nguyên; cạnh từ `x`: tới mọi bội `x·k ≤ CAP` (`k ≥ 2`) và tới mọi `x / k` với `2 ≤ k < x`, `k | x`.
3. Với mỗi giá trị ban đầu `a` chạy BFS để có `dist[a][t]` = số phép ít nhất tới `t`.
4. Đáp án = `min` theo đích `t` của `Σ dist[a_i][t]`.

Vì mọi giá trị nhỏ nên `lcm` của chúng nhỏ, đích tối ưu và các đỉnh trung gian cần thiết (tối đa `a·t`) đều nằm trong `CAP`. Ta dựng bảng ước một lần để BFS nhanh.

In [6]:
from functools import lru_cache

CAP = 8000
proper_divs = [[] for _ in range(CAP + 1)]       # proper_divs[m] = các ước d với 2 <= d < m
for d in range(2, CAP + 1):
    for m in range(2 * d, CAP + 1, d):
        proper_divs[m].append(d)

@lru_cache(None)
def bfs_dist(a):
    dist = {a: 0}
    q = deque([a])
    while q:
        x = q.popleft()
        nxt = list(range(2 * x, CAP + 1, x))      # nhân với k >= 2
        nxt += proper_divs[x]                      # chia cho k (kết quả là ước d >= 2, d < x)
        for y in nxt:
            if y not in dist:
                dist[y] = dist[x] + 1
                q.append(y)
    return dist

def brute_force(nums):
    tables = [bfs_dist(a) for a in nums]
    best = INF
    for t in range(1, CAP + 1):
        s = 0
        for d in tables:
            if t not in d:
                s = INF
                break
            s += d[t]
        best = min(best, s)
    return best

for nums, expected in EXAMPLES:
    print(nums, "->", brute_force(nums), "(kỳ vọng", expected, ")")

[6, 12, 8] -> 3 (kỳ vọng 3 )
[5, 15, 20] -> 2 (kỳ vọng 2 )
[7, 7, 7] -> 0 (kỳ vọng 0 )


Kiểm chứng **công thức chi phí từng cặp** `pair_cost` bằng BFS cho mọi `a, t ≤ 60`:

In [7]:
for a in range(1, 61):
    d = bfs_dist(a)
    for t in range(1, 61):
        assert d.get(t, INF) == pair_cost(a, t), (a, t, d.get(t), pair_cost(a, t))
print("pair_cost khớp BFS với mọi a, t ≤ 60")

pair_cost khớp BFS với mọi a, t ≤ 60


**Độ phức tạp.** Mỗi BFS: `O(CAP log CAP)`; số lần lặp theo đích: `O(CAP · n)`. Chỉ dùng cho giá trị rất nhỏ — nhưng đủ làm **trọng tài**. **Nút thắt:** đích có thể lên tới rất lớn (bội chung), nên không thể quét mọi đích. Ta cần biết đích nào đáng xét.

## 6. Bước đột phá — chỉ cần xét đích là phần tử của mảng

Biết `cost(a, t)`, tổng chi phí cho đích `t` (với `t ≥ 2`) là: elements bằng `t` tốn 0; elements **có liên hệ ước/bội** với `t` tốn 1; elements **không liên hệ** tốn 2.

Gọi `cnt[t]` = số phần tử bằng `t`, `A(t)` = số phần tử `< t` là ước của `t`, `B(t)` = số phần tử `> t` là bội của `t`. Khi đó:

`cost(t) = 0·cnt[t] + 1·(A + B) + 2·(n − cnt[t] − A − B) = 2n − 2·cnt[t] − A(t) − B(t)`.

**Quan sát 1 (đích rất lớn).** Với `T = lcm(tất cả)` (hoặc bội bất kỳ của nó), mọi phần tử là ước của `T` nên mỗi phần tử tốn `≤ 1` → tổng `≤ n`. Đây là phương án nền: **đáp án luôn `≤ n`** (khi không phải toàn số 1).

**Quan sát 2 (đích không phải phần tử).** Nếu `t` không bằng phần tử nào thì `cnt[t] = 0` và `cost(t) = 2n − A − B ≥ n` (vì `A + B ≤ n`). Nó không bao giờ tốt hơn phương án nền `n`.

**Kết luận.** Chỉ có hai loại ứng viên:

1. Đích rất lớn: chi phí `n`.
2. Đích bằng một phần tử `t ≥ 2`: chi phí `2n − 2·cnt[t] − A(t) − B(t)`.

Đáp án = `min` của tất cả. Trường hợp đặc biệt: toàn số `1` → đáp án `0`. Và đích `t = 1` chỉ hợp lệ khi toàn số `1` (vì số khác `1` không về được `1`), nên ta **bỏ qua** `t = 1` trong vòng lặp đích.

Hãy kiểm chứng công thức này (cách 2 bên dưới) trước khi tối ưu.

## 7. Các cách tiếp theo

Từ công thức trên, ba cách khác nhau tính `A(t)`, `B(t)`:

- **Cách 2:** so sánh mọi cặp `(a, t)` bằng `%` — `O(m²)`.
- **Cách 3:** **sàng bội** khi giá trị `≤ V` nhỏ — `O(V log V)`.
- **Cách 4:** phân tích thừa số mỗi phần tử rồi **liệt kê ước** — chạy cho giá trị tới `10^9` (đáp án nộp).
- **Cách 5:** như cách 4 nhưng phân tích thừa số hàng loạt bằng `numpy` để tăng tốc.

### 7.1. Cách 2 — So sánh mọi cặp, `O(m²)`

**Giải thích từng đoạn.**

1. `cnt = Counter(nums)`; nếu chỉ toàn số `1` thì trả `0`.
2. `best = n` (phương án đích lớn).
3. Với mỗi giá trị phân biệt `t ≥ 2`, cộng `cnt[a] · pair_cost(a, t)` qua mọi giá trị phân biệt `a`.
4. `min`.

In [8]:
def by_pairs(nums):
    n = len(nums)
    cnt = Counter(nums)
    if set(cnt) == {1}:
        return 0
    best = n
    for t in cnt:
        if t == 1:
            continue
        best = min(best, sum(c * pair_cost(a, t) for a, c in cnt.items()))
    return best

for nums, expected in EXAMPLES:
    assert by_pairs(nums) == expected
print("Cách 2 khớp ví dụ")

Cách 2 khớp ví dụ


**Độ phức tạp.** `O(m²)` với `m` = số giá trị phân biệt. Với `m = 10^5` là `10^10` — quá chậm, nhưng chính xác làm trọng tài thứ hai.

### 7.2. Cách 3 — Sàng bội (khi giá trị `≤ V`, `V` không quá lớn)

**Ý tưởng.** Thay vì với mỗi `t` kiểm tra mọi `a`, ta đi theo kiểu **sàng Eratosthenes**: với mỗi `d` từ `1` tới `V`, nhảy qua các bội `m = 2d, 3d, ...`:

- `A[m] += cnt[d]` (`d` là ước nhỏ hơn của `m`),
- `B[d] += cnt[m]` (`m` là bội lớn hơn của `d`).

Tổng số bước là `V/1 + V/2 + ... ≈ V ln V` (chuỗi điều hoà).

In [9]:
def by_sieve(nums):
    n = len(nums)
    V = max(nums)
    cnt = [0] * (V + 1)
    for x in nums:
        cnt[x] += 1
    if cnt[1] == n:
        return 0
    A = [0] * (V + 1)
    B = [0] * (V + 1)
    for d in range(1, V + 1):
        if cnt[d] == 0:
            continue
        for m in range(2 * d, V + 1, d):
            if cnt[m]:
                A[m] += cnt[d]
                B[d] += cnt[m]
    best = n
    for t in range(2, V + 1):
        if cnt[t]:
            best = min(best, 2 * n - 2 * cnt[t] - A[t] - B[t])
    return best

for nums, expected in EXAMPLES:
    assert by_sieve(nums) == expected
print("Cách 3 khớp ví dụ")

Cách 3 khớp ví dụ


**Độ phức tạp.** `O(V log V)` thời gian, `O(V)` bộ nhớ. Rất hợp khi `V ≤ 10^6`; với `V = 10^9` thì không thể (mảng quá lớn).

### 7.3. Cách 4 — Phân tích thừa số + liệt kê ước (đáp án nộp)

**Ý tưởng.** Với giá trị tới `10^9`, ta không sàng được nhưng một số `m ≤ 10^9` có **tối đa 1344 ước**. Với mỗi giá trị phân biệt `m`:

1. Phân tích `m` ra thừa số nguyên tố bằng chia thử cho các số nguyên tố `p ≤ √m` (sàng sẵn các số nguyên tố tới `√max`).
2. Sinh mọi ước `d` của `m` từ phân tích đó.
3. Với mỗi ước `d ≠ m` có trong `cnt`: `A[m] += cnt[d]` và `B[d] += cnt[m]`.

Mỗi cặp (ước, bội) trong mảng được duyệt đúng **một lần** — từ phía bội — nên không cần so sánh `m²` cặp.

**Giải thích từng đoạn của hàm phân tích.**

- `for p in primes: if p*p > x: break` — khi `p² > x` thì phần còn lại `x` (nếu `> 1`) là số nguyên tố.
- Mỗi lần gặp `p | x`, chia hết mọi luỹ thừa của `p`, ghi số mũ `e`.
- Sinh ước: bắt đầu `divs = [1]`; với mỗi `(p, e)` nhân mọi ước hiện có với `p^0, p^1, ..., p^e`.

In [10]:
def primes_upto(limit):
    sieve = bytearray([1]) * (limit + 1)
    sieve[0:2] = b"\x00\x00"
    for i in range(2, isqrt(limit) + 1):
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(range(i * i, limit + 1, i)))
    return [i for i in range(2, limit + 1) if sieve[i]]

def divisors_of(m, primes):
    divs = [1]
    x = m
    for p in primes:
        if p * p > x:
            break
        if x % p == 0:
            e = 0
            while x % p == 0:
                x //= p
                e += 1
            pw = [p ** i for i in range(e + 1)]
            divs = [d * q for d in divs for q in pw]
    if x > 1:                                # còn một thừa số nguyên tố lớn
        divs += [d * x for d in divs]
    return divs

primes_demo = primes_upto(100)
print(sorted(divisors_of(360, primes_demo)))                # 24 ước
print(len(divisors_of(735134400, primes_upto(isqrt(735134400) + 1))))   # 1344 ước (số có nhiều ước nhất ≤ 1e9)
print(sorted(divisors_of(97, primes_demo)))                 # số nguyên tố

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 18, 20, 24, 30, 36, 40, 45, 60, 72, 90, 120, 180, 360]
1344
[1, 97]


In [11]:
def by_factorization(nums):
    n = len(nums)
    cnt = Counter(nums)
    if set(cnt) == {1}:
        return 0
    primes = primes_upto(isqrt(max(nums)) + 1)
    A = defaultdict(int)
    B = defaultdict(int)
    for m in cnt:
        for d in divisors_of(m, primes):
            if d != m and d in cnt:
                A[m] += cnt[d]               # d là ước nhỏ hơn của m
                B[d] += cnt[m]               # m là bội lớn hơn của d
    best = n
    for t, c in cnt.items():
        if t == 1:
            continue
        best = min(best, 2 * n - 2 * c - A[t] - B[t])
    return best

for nums, expected in EXAMPLES:
    assert by_factorization(nums) == expected
print("Cách 4 khớp ví dụ")

Cách 4 khớp ví dụ


**Độ phức tạp.** Gọi `m` = số giá trị phân biệt, `V = max(nums)`, `P = π(√V) ≈ 3400` khi `V = 10^9`, `D ≤ 1344` = số ước tối đa. Phân tích thừa số tốn `O(P)` mỗi số (trường hợp xấu nhất: số nguyên tố lớn), sinh ước và tra `cnt` tốn `O(D)`. Tổng `O(n + m·(P + D))` thời gian, `O(m + P)` bộ nhớ. Trung bình rất nhỏ vì số ước trung bình của số ngẫu nhiên chỉ cỡ `ln V ≈ 20`.

### 7.4. Cách 5 — Phân tích thừa số hàng loạt bằng `numpy`

**Ý tưởng.** Phần tốn nhất của cách 4 (nhất là với dữ liệu có nhiều số nguyên tố lớn) là vòng chia thử `P` số nguyên tố cho **mỗi** số bằng Python thuần. Ta đảo vòng lặp: với **mỗi số nguyên tố `p`**, dùng `numpy` kiểm tra `arr % p == 0` cho **cả mảng** một lần. Chỉ các vị trí chia hết mới cần xử lý bằng Python.

In [12]:
import numpy as np

def factorize_batch(values, primes):
    """Trả về list các list [(p, e), ...] cho từng giá trị (dùng numpy để lọc nhanh)."""
    arr = np.array(values, dtype=np.int64)
    rem = arr.copy()
    facs = [[] for _ in values]
    for p in primes:
        idx = np.nonzero(rem % p == 0)[0]
        for i in idx.tolist():
            x = int(rem[i])
            e = 0
            while x % p == 0:
                x //= p
                e += 1
            rem[i] = x
            facs[i].append((p, e))
    for i, x in enumerate(rem.tolist()):
        if x > 1:
            facs[i].append((x, 1))
    return facs

def divisors_from_factors(f):
    divs = [1]
    for p, e in f:
        pw = [p ** i for i in range(e + 1)]
        divs = [d * q for d in divs for q in pw]
    return divs

def by_factorization_numpy(nums):
    n = len(nums)
    cnt = Counter(nums)
    if set(cnt) == {1}:
        return 0
    values = list(cnt)
    primes = primes_upto(isqrt(max(nums)) + 1)
    facs = factorize_batch(values, primes)
    A = defaultdict(int)
    B = defaultdict(int)
    for m, f in zip(values, facs):
        for d in divisors_from_factors(f):
            if d != m and d in cnt:
                A[m] += cnt[d]
                B[d] += cnt[m]
    best = n
    for t, c in cnt.items():
        if t != 1:
            best = min(best, 2 * n - 2 * c - A[t] - B[t])
    return best

for nums, expected in EXAMPLES:
    assert by_factorization_numpy(nums) == expected
print("Cách 5 khớp ví dụ")

Cách 5 khớp ví dụ


**Độ phức tạp.** Cùng bậc `O(m·P)` (mỗi số nguyên tố quét cả mảng), nhưng phần chia thử được vector hoá nên hằng số khác; lợi ích phụ thuộc dữ liệu. Ta đo thời gian thật ở mục 11 thay vì khẳng định trước.

## 8. TRACE từng bước

Ta theo dõi cách 4 trên `nums = [6, 12, 8]` và thêm một ví dụ có số `1`. Với mỗi giá trị phân biệt `m`, in các ước nằm trong mảng; sau đó in bảng `cnt, A, B, score, cost` cho mỗi đích.

| cột | ý nghĩa |
|---|---|
| `cnt` | số phần tử bằng `t` |
| `A` | số phần tử nhỏ hơn là ước của `t` |
| `B` | số phần tử lớn hơn là bội của `t` |
| `cost` | `2n − 2cnt − A − B` |

In [13]:
def trace_solution(nums):
    n = len(nums)
    cnt = Counter(nums)
    print("nums =", nums, " n =", n, " cnt =", dict(cnt))
    if set(cnt) == {1}:
        print("toàn số 1 -> 0"); return 0
    primes = primes_upto(isqrt(max(nums)) + 1)
    A, B = defaultdict(int), defaultdict(int)
    for m in cnt:
        divs = sorted(divisors_of(m, primes))
        hit = [d for d in divs if d != m and d in cnt]
        print(f"  m={m:>3}: ước {divs}  -> ước có trong mảng {hit}")
        for d in hit:
            A[m] += cnt[d]; B[d] += cnt[m]
    print(f"\n  {'t':>3} {'cnt':>4} {'A':>3} {'B':>3} {'cost=2n-2cnt-A-B':>18}")
    best = n
    for t, c in cnt.items():
        if t == 1:
            print(f"  {t:>3}   (bỏ qua: không ai khác về được 1)")
            continue
        cost = 2 * n - 2 * c - A[t] - B[t]
        print(f"  {t:>3} {c:>4} {A[t]:>3} {B[t]:>3} {cost:>18}")
        best = min(best, cost)
    print("  phương án đích rất lớn (lcm):", n)
    print("  => đáp án", best)
    return best

trace_solution([6, 12, 8])
print()
trace_solution([1, 1, 3, 6])

nums = [6, 12, 8]  n = 3  cnt = {6: 1, 12: 1, 8: 1}
  m=  6: ước [1, 2, 3, 6]  -> ước có trong mảng []
  m= 12: ước [1, 2, 3, 4, 6, 12]  -> ước có trong mảng [6]
  m=  8: ước [1, 2, 4, 8]  -> ước có trong mảng []

    t  cnt   A   B   cost=2n-2cnt-A-B
    6    1   0   1                  3
   12    1   1   0                  3
    8    1   0   0                  4
  phương án đích rất lớn (lcm): 3
  => đáp án 3

nums = [1, 1, 3, 6]  n = 4  cnt = {1: 2, 3: 1, 6: 1}
  m=  1: ước [1]  -> ước có trong mảng []
  m=  3: ước [1, 3]  -> ước có trong mảng [1]
  m=  6: ước [1, 2, 3, 6]  -> ước có trong mảng [1, 3]

    t  cnt   A   B   cost=2n-2cnt-A-B
    1   (bỏ qua: không ai khác về được 1)
    3    1   2   1                  3
    6    1   3   0                  3
  phương án đích rất lớn (lcm): 4
  => đáp án 3


3

Với `[6, 12, 8]`: đích `6` có `A = 0`, `B = 1` (12 là bội) → `2·3 − 2 − 0 − 1 = 3`. Đích `12`: `A = 1` (6), → `6 − 2 − 1 − 0 = 3`. Đích `8`: không liên hệ ai → `6 − 2 = 4`. Phương án đích lớn: 3. Đáp án 3. Ở ví dụ thứ hai, đích `1` bị bỏ qua dù `cnt[1] = 2`; nếu không bỏ, ta sẽ ra đáp án sai nhỏ hơn.

## 9. Code nộp LeetCode

Tên phương thức dùng dự đoán `minOperations` (bài Premium, cần đối chiếu lại).

In [14]:
from typing import List
from collections import Counter, defaultdict
from math import isqrt

class Solution:
    def minOperations(self, nums: List[int]) -> int:
        n = len(nums)
        cnt = Counter(nums)
        if len(cnt) == 1 and 1 in cnt:           # toàn số 1
            return 0

        # sàng các số nguyên tố tới căn của giá trị lớn nhất
        limit = isqrt(max(nums)) + 1
        sieve = bytearray([1]) * (limit + 1)
        sieve[0:2] = b"\x00\x00"
        for i in range(2, isqrt(limit) + 1):
            if sieve[i]:
                sieve[i * i::i] = bytearray(len(range(i * i, limit + 1, i)))
        primes = [i for i in range(2, limit + 1) if sieve[i]]

        A = defaultdict(int)   # A[m] = số phần tử nhỏ hơn là ước của m
        B = defaultdict(int)   # B[d] = số phần tử lớn hơn là bội của d
        for m in cnt:
            divs = [1]
            x = m
            for p in primes:
                if p * p > x:
                    break
                if x % p == 0:
                    e = 0
                    while x % p == 0:
                        x //= p
                        e += 1
                    pw = [p ** i for i in range(e + 1)]
                    divs = [d * q for d in divs for q in pw]
            if x > 1:
                divs += [d * x for d in divs]
            for d in divs:
                if d != m and d in cnt:
                    A[m] += cnt[d]
                    B[d] += cnt[m]

        best = n                                   # đích là bội chung rất lớn
        for t, c in cnt.items():
            if t == 1:
                continue                           # không ai về được 1
            best = min(best, 2 * n - 2 * c - A[t] - B[t])
        return best

for nums, expected in EXAMPLES:
    assert Solution().minOperations(nums) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (chi phí cặp).** Với `a, t ≥ 2`:

- `a = t`: 0 phép.
- `t | a`, `t < a`: chia `a` cho `k = a / t`, thoả `2 ≤ k < a` (vì `t ≥ 2` nên `k < a`; vì `t < a` nên `k ≥ 2`) → 1 phép.
- `a | t`, `a < t`: nhân với `k = t / a ≥ 2` → 1 phép.
- Ngược lại: không thể 1 phép (một phép chỉ cho bội hoặc ước của `a`), nhưng `a → a·t → t` hợp lệ (nhân với `t ≥ 2`; chia `a·t` cho `a` thoả `2 ≤ a < a·t`, `a | a·t`) → đúng 2 phép.

Với `a = 1`: `1 | t` nên 1 phép (nhân với `t`), và `t = 1` chỉ đạt khi `a = 1`. Với `t = 1` và `a ≥ 2`: nhân chỉ tăng, chia luôn cho kết quả `≥ 2`, nên **không bao giờ** về `1`. BFS đã xác nhận bảng này ở mục 5.

**Bổ đề 2 (chọn đích).** Các phần tử độc lập nhau nên tổng chi phí đích `t` là tổng chi phí từng phần tử. Với `t` không phải phần tử: `cost ≥ n`; đích `T = lcm(...)` luôn đạt `≤ n`. Vậy đích không thuộc mảng không bao giờ thắng được `n`.

**Bổ đề 3 (công thức).** Chi phí đích phần tử `t ≥ 2` là `2n − 2cnt[t] − A(t) − B(t)`, đã chứng minh ở mục 6 bằng cách đếm số phần tử theo ba loại chi phí.

**Bổ đề 4 (tính `A`, `B`).** Mỗi cặp `(d, m)` với `d | m`, `d < m`, cùng thuộc mảng được duyệt đúng một lần khi xử lý `m` (liệt kê ước của `m`), đóng góp `cnt[d]` cho `A[m]` và `cnt[m]` cho `B[d]`. Hai bảng này đầy đủ và không trùng.

**Trường hợp biên.** Toàn số `1` → `0`. `n = 1` → `best = min(1, 2 − 2 − 0 − 0) = 0`. Số `1` lẫn các số khác: `1` là ước của mọi `t ≥ 2` nên đóng góp vào `A[t]`, đúng như chi phí 1 (nhân với `t`).

## 11. Phân tích Time/Space

Gọi `n = len(nums)`, `m` = số giá trị phân biệt, `V = max(nums)`.

| Cách | Time | Space |
|---|---|---|
| Brute force BFS (chỉ giá trị nhỏ) | `O(CAP log CAP + CAP·n)` | `O(CAP log CAP)` |
| So sánh mọi cặp | `O(m²)` | `O(m)` |
| Sàng bội | `O(V log V)` | `O(V)` |
| Phân tích thừa số + liệt kê ước (nộp) | `O(n + m·(π(√V) + D))`, `D ≤ 1344` | `O(m + √V / ln V)` |
| Như trên, vector hoá bằng numpy | cùng bậc `O(m·P)`, hằng số khác (xem số đo) | `O(m + √V / ln V)` |

Đo thực tế trên dữ liệu ngẫu nhiên (số lượng nhỏ cho Python thuần, lớn cho numpy):

In [15]:
import time, random

rng = random.Random(2024)
small = [rng.randint(1, 10**9) for _ in range(3000)]

t0 = time.time(); r1 = by_factorization(small); t1 = time.time() - t0
t0 = time.time(); r2 = by_factorization_numpy(small); t2 = time.time() - t0
assert r1 == r2 == Solution().minOperations(small)
print(f"n=3000 (giá trị ≤ 1e9): Python thuần {t1:.2f}s, numpy {t2:.2f}s, đáp án {r1}")

big = [rng.randint(1, 10**9) for _ in range(100000)]
t0 = time.time(); rb1 = Solution().minOperations(big); tb1 = time.time() - t0
t0 = time.time(); rb2 = by_factorization_numpy(big); tb2 = time.time() - t0
assert rb1 == rb2
print(f"n=1e5 ngẫu nhiên: code nộp (Python thuần) {tb1:.2f}s, numpy {tb2:.2f}s, đáp án {rb1} (≤ n)")

# Dữ liệu "xấu": 1e5 số nguyên tố lớn gần 1e9 -> mỗi số chia thử gần hết 3401 số nguyên tố (chỉ chạy 3000 số)
primes_big = primes_upto(10**5)
cands = [x for x in range(10**9 - 40000, 10**9) if all(x % p for p in primes_big[:3401] if p * p <= x)]
sub = cands[:3000]
t0 = time.time(); Solution().minOperations(sub); tw = time.time() - t0
print(f"{len(sub)} số nguyên tố ~1e9 (xấu nhất cho chia thử): {tw:.2f}s  ->  ước lượng 1e5 số: {tw * 100000 / len(sub):.0f}s")

n=3000 (giá trị ≤ 1e9): Python thuần 0.08s, numpy 0.05s, đáp án 3000


n=1e5 ngẫu nhiên: code nộp (Python thuần) 2.26s, numpy 1.69s, đáp án 100000 (≤ n)


1928 số nguyên tố ~1e9 (xấu nhất cho chia thử): 0.29s  ->  ước lượng 1e5 số: 15s


## 12. Bộ test

Gồm ví dụ đề, edge case, phản ví dụ, rồi đối chiếu ngẫu nhiên giữa BFS (định nghĩa), cách 2, cách 3, cách 4, cách 5.

In [16]:
S = Solution()
for nums, expected in EXAMPLES:
    for fn in (brute_force, by_pairs, by_sieve, by_factorization, by_factorization_numpy, S.minOperations):
        assert fn(nums) == expected, (fn.__name__, nums)
print("Ví dụ đề: tất cả các cách khớp")

Ví dụ đề: tất cả các cách khớp


In [17]:
# Edge case
assert S.minOperations([1]) == 0
assert S.minOperations([1, 1, 1]) == 0            # toàn số 1
assert S.minOperations([1, 5]) == 1               # 1 -> 5 bằng một phép nhân
assert S.minOperations([1, 1, 3]) == 2            # KHÔNG phải 1: không ai về được 1
assert S.minOperations([2, 3]) == 2               # 2 -> 6, 3 -> 6 hoặc đường vòng
assert S.minOperations([2, 3, 5]) == 3            # đích rất lớn 30 tốn 3, tốt hơn mọi đích phần tử
assert S.minOperations([12, 18, 30]) == 3         # đích 6 (không phải phần tử) cũng chỉ tốn 3
assert S.minOperations([4, 6, 12]) == 2           # đích 12: 4->12, 6->12
assert S.minOperations([10**9, 10**9]) == 0
assert S.minOperations([999999937, 2]) == 2       # số nguyên tố lớn và 2: unrelated -> 2
print("OK")

OK


Thêm một trường hợp hay nhầm: `[2, 2, 3]`. Đích `2` tốn `0 + 0 + 2 = 2`; đích `3` tốn `2 + 2 + 0 = 4`; đích lớn `6` tốn `1 + 1 + 1 = 3`. Vậy đáp án là `2` (không phải `1`). Ta xác nhận bằng brute force (bài học: luôn đối chiếu BFS thay vì tin trực giác).

In [18]:
print(S.minOperations([2, 2, 3]), brute_force([2, 2, 3]))
assert S.minOperations([2, 2, 3]) == brute_force([2, 2, 3]) == 2

2 2


**Phản ví dụ cho các cách làm sai** (xem thêm mục 13): quên bỏ đích `1`, quên phương án đích lớn, quên hướng chia. Ví dụ chạy tay ở mục 13.

In [19]:
# Đối chiếu ngẫu nhiên có seed với BFS (định nghĩa)
rng = random.Random(12345)
for it in range(350):
    n = rng.randint(1, 5)
    hi = rng.choice([3, 4, 6, 8])
    nums = [rng.randint(1, hi) for _ in range(n)]
    ref = brute_force(nums)
    assert by_pairs(nums) == ref, nums
    assert by_sieve(nums) == ref, nums
    assert by_factorization(nums) == ref, nums
    assert by_factorization_numpy(nums) == ref, nums
    assert S.minOperations(nums) == ref, nums
print("350 input ngẫu nhiên nhỏ: cả 5 cách khớp BFS")

350 input ngẫu nhiên nhỏ: cả 5 cách khớp BFS


In [20]:
# Input trung bình: các cách khớp nhau (BFS không với tới)
rng = random.Random(777)
for it in range(250):
    n = rng.randint(1, 40)
    hi = rng.choice([12, 30, 60, 120, 500])
    nums = [rng.randint(1, hi) for _ in range(n)]
    ref = by_pairs(nums)
    assert by_sieve(nums) == ref and by_factorization(nums) == ref
    assert by_factorization_numpy(nums) == ref and S.minOperations(nums) == ref
print("250 input trung bình: khớp")

# Tập có cấu trúc: toàn bộ ước của 720720 cộng thêm bội và số ngẫu nhiên lớn
N = 720720
divs = [d for d in range(1, N + 1) if N % d == 0]
nums = divs + [N * 3, N * 7, 1] + [rng.randint(1, 10**9) for _ in range(50)]
rng.shuffle(nums)
assert by_pairs(nums) == S.minOperations(nums) == by_factorization_numpy(nums)
print("tập ước của 720720:", len(divs), "ước; đáp án", S.minOperations(nums))

250 input trung bình: khớp
tập ước của 720720: 240 ước; đáp án 293


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên rằng không thể về `1` (đích `t = 1` không hợp lệ)

Công thức `2n − 2cnt − A − B` áp dụng cho `t = 1` cho giá trị nhỏ vì mọi số đều là bội của `1`. Nhưng thực tế số lớn hơn `1` không bao giờ chia về `1` (đề chặn `k < x`).

In [21]:
def wrong_allow_one(nums):
    n = len(nums); cnt = Counter(nums)
    best = n
    for t, c in cnt.items():                      # SAI: không bỏ t == 1
        A = sum(cnt[a] for a in cnt if a < t and t % a == 0)
        B = sum(cnt[m] for m in cnt if m > t and m % t == 0)
        best = min(best, 2 * n - 2 * c - A - B)
    return best

print("sai :", wrong_allow_one([1, 1, 3]))           # 1 (nhưng 3 không về 1 được)
print("đúng:", Solution().minOperations([1, 1, 3]))  # 2

sai : 1
đúng: 2


**Cách đúng:** bỏ qua `t = 1` (chỉ trả `0` khi toàn số `1`). Phương án "đích lớn" vẫn dùng được: `1 → 3` (nhân), `3` giữ nguyên.

### Lỗi 2 — Quên phương án "đích là bội chung lớn" (chỉ xét đích là phần tử)

Với `[2, 3, 5]` không có phần tử nào liên hệ với phần tử khác. Mọi đích phần tử tốn `2n − 2 = 4`, nhưng đích `30` (không có trong mảng) chỉ tốn 3.

In [22]:
def wrong_only_elements(nums):
    n = len(nums); cnt = Counter(nums)
    if set(cnt) == {1}: return 0
    best = INF                                     # SAI: thiếu best = n
    for t, c in cnt.items():
        if t == 1: continue
        A = sum(cnt[a] for a in cnt if a < t and t % a == 0)
        B = sum(cnt[m] for m in cnt if m > t and m % t == 0)
        best = min(best, 2 * n - 2 * c - A - B)
    return best

print("sai :", wrong_only_elements([2, 3, 5]))        # 4
print("đúng:", Solution().minOperations([2, 3, 5]))   # 3

sai : 4
đúng: 3


**Cách đúng:** khởi tạo `best = n` (đích là bội chung lớn).

### Lỗi 3 — Chỉ tính hướng **nhân** (ước), quên hướng **chia** (bội)

Nghĩ rằng chỉ phần tử nào là ước của đích mới tốn 1 phép; quên rằng phần tử là **bội** của đích cũng tốn 1 (chia một lần).

In [23]:
def wrong_only_divisors(nums):
    n = len(nums); cnt = Counter(nums)
    if set(cnt) == {1}: return 0
    best = n
    for t, c in cnt.items():
        if t == 1: continue
        A = sum(cnt[a] for a in cnt if a < t and t % a == 0)   # chỉ ước
        best = min(best, 2 * n - 2 * c - A)                    # SAI: thiếu B
    return best

print("sai :", wrong_only_divisors([5, 15, 20]))        # 4 - vì không đếm 15, 20 là bội của 5
print("đúng:", Solution().minOperations([5, 15, 20]))    # 2

sai : 3
đúng: 2


**Cách đúng:** cộng cả `B(t)` (số bội lớn hơn trong mảng).

### Lỗi 4 — Đếm phần tử bằng `t` vào `A` hoặc `B`

Nếu liệt kê ước `d` mà không loại `d == m`, phần tử bằng đích bị tính vừa là "0 phép" vừa là "1 phép".

In [24]:
def wrong_count_self(nums):
    n = len(nums); cnt = Counter(nums)
    if set(cnt) == {1}: return 0
    best = n
    for t, c in cnt.items():
        if t == 1: continue
        A = sum(cnt[a] for a in cnt if a <= t and t % a == 0)   # SAI: a <= t gồm cả a == t
        B = sum(cnt[m] for m in cnt if m > t and m % t == 0)
        best = min(best, 2 * n - 2 * c - A - B)
    return best

print("sai :", wrong_count_self([6, 6, 12]))       # -1: chi phí âm là vô lý
print("đúng:", Solution().minOperations([6, 6, 12]))

sai : -1
đúng: 1


**Cách đúng:** ước `d` phải **nhỏ hơn hẳn** `m` (`d != m`); cặp bằng nhau đã tính qua `cnt[t]`.

## 14. Mẹo & nhận dạng pattern

- **"Nhân hoặc chia cho số nguyên bất kỳ"** → mỗi phép nhảy được tới **mọi bội/ước**, nên số phép rất nhỏ (≤ 2). Hãy thử tính chi phí cặp trước khi nghĩ tới BFS.
- **Đưa bài toán về đếm ước/bội** khi chi phí chỉ phụ thuộc quan hệ chia hết: dùng `Counter` + sàng bội hoặc liệt kê ước.
- **Giá trị ≤ 10^9**: phân tích thừa số bằng chia thử tới `√V ≈ 31623` (3401 số nguyên tố) và liệt kê ước (≤ 1344).
- **Giá trị ≤ 10^6**: sàng bội `O(V log V)` thường đơn giản và nhanh hơn.
- Luôn tự hỏi "đích tối ưu nằm trong tập nào?" — ở đây là phần tử của mảng hoặc "bội chung lớn".
- Với bài Premium không có code mẫu, hãy tự xây brute force theo đúng định nghĩa đề để kiểm chứng.

## 15. Biến thể

1. **Chỉ được nhân**: mọi phần tử phải lên cùng một bội chung. Đáp án = `n − max(cnt[t])` trên các `t` là bội chung của toàn mảng (hoặc `n` nếu không có).
2. **Chỉ được chia**: đích là ước chung `≥ 2`, nếu không có ước chung `≥ 2` thì không thể (trừ khi bằng nhau sẵn).
3. **Chi phí khác nhau**: nhân tốn `x`, chia tốn `y`: chi phí cặp thay đổi, đường vòng `a → a·t → t` tốn `x + y`.
4. **Chỉ nhân/chia cho số nguyên tố**: bài trở thành đếm số mũ thừa số nguyên tố.

Ta cài thử biến thể 1 và kiểm chứng bằng BFS chỉ có cạnh nhân.

In [25]:
def multiply_only(nums):
    n = len(nums)
    cnt = Counter(nums)
    L = lcm(*cnt)
    best = n
    for t, c in cnt.items():
        if t % L == 0:                # t là bội chung của cả mảng
            best = min(best, n - c)
    return best

@lru_cache(None)
def bfs_mul_only(a):
    dist = {a: 0}
    q = deque([a])
    while q:
        x = q.popleft()
        for y in range(2 * x, CAP + 1, x):
            if y not in dist:
                dist[y] = dist[x] + 1
                q.append(y)
    return dist

def multiply_only_brute(nums):
    tables = [bfs_mul_only(a) for a in nums]
    return min(sum(d.get(t, INF) for d in tables) for t in range(1, CAP + 1))

rng = random.Random(5)
for _ in range(200):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(1, 4))]
    assert multiply_only(nums) == multiply_only_brute(nums), nums
print("Biến thể chỉ-nhân: khớp BFS;", multiply_only([2, 3, 6]), multiply_only([2, 4, 4]))

Biến thể chỉ-nhân: khớp BFS; 2 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS vét cạn | BFS từng phần tử, thử mọi đích ≤ CAP | `O(CAP log CAP + CAP·n)` | `O(CAP log CAP)` | trọng tài, giá trị ≤ 8 |
| 2. So sánh mọi cặp | `pair_cost` cho mọi đích phần tử | `O(m²)` | `O(m)` | `m ≤ 3000`, trọng tài thứ hai |
| 3. Sàng bội | `A`, `B` bằng nhảy bội | `O(V log V)` | `O(V)` | `V ≤ 10^6` |
| 4. Thừa số + liệt kê ước (nộp) | mỗi cặp ước-bội đếm một lần | `O(n + m(P + D))` | `O(m + P)` | `V ≤ 10^9` |
| 5. Như 4, numpy | lọc chia hết hàng loạt | cùng bậc, hằng số khác | `O(m + P)` | muốn thử vector hoá; đo trước khi chọn |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `pair_cost(8, 6)`, `pair_cost(6, 12)`, `pair_cost(3, 1)` bằng bao nhiêu?

**Bài 2 (tính tay).** `nums = [4, 6, 12]`. Tính tổng chi phí cho đích `12`, `4`, `6`, rồi cho đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `pair_cost_todo(a, t)` (cùng quy tắc như `pair_cost`).

**Bài 4 (tự viết lại).** Viết `my_solve(nums)` bằng cách **so sánh mọi cặp** (không nhìn code nộp).

**Bài 5 (biến thể / mini challenge).** Nếu chia tốn `y` điểm, nhân tốn `x` điểm, `pair_cost_xy(a, t, x, y)` là gì (giả sử `a, t ≥ 2`)? Viết hàm và kiểm tra bằng BFS có trọng số (Dijkstra) trên giá trị nhỏ.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def pair_cost_todo(a, t):
    # TODO 1: a == t -> 0
    # TODO 2: t == 1 -> INF (chỉ a == 1 mới về được 1)
    # TODO 3: một trong hai chia hết cái kia -> 1
    # TODO 4: còn lại -> 2
    pass

# Bài 4 — TODO
def my_solve(nums):
    # TODO: Counter; toàn số 1 -> 0; best = n; với mỗi đích t >= 2 trong cnt, tính tổng chi phí
    pass

# Bài 5 — TODO
def pair_cost_xy(a, t, x, y):
    # TODO: nhân tốn x, chia tốn y; a, t >= 2
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: kiểm tra `a == t`, rồi quan hệ chia hết.
- Mức 2: `8` và `6` không chia hết nhau.
- Mức 3: `2`, `1`, `INF` (không về được 1).

**Bài 2.**
- Mức 1: dùng `cost(a, t)`.
- Mức 2: đích 12: `4 → 12` (1), `6 → 12` (1), `12 → 12` (0).
- Mức 3: đích 12 tốn 2; đích 4: `0 + 2 + 1 = 3`; đích 6: `2 + 0 + 1 = 3`; đích lớn tốn 3 → đáp án `2`.

**Bài 3.**
- Mức 1: bốn nhánh `if`.
- Mức 2: `a % t == 0 or t % a == 0`.
- Mức 3: thứ tự: `a == t`, `t == 1`, chia hết, còn lại.

**Bài 4.**
- Mức 1: `best = n`.
- Mức 2: `sum(c * pair_cost(a, t) for a, c in cnt.items())`.
- Mức 3: bỏ qua `t == 1`.

**Bài 5.**
- Mức 1: cặp có quan hệ: nhân tốn `x`, chia tốn `y`.
- Mức 2: không quan hệ: `t` không là bội của `a` nên cần ít nhất một phép chia; `t` không là ước của `a` nên cần ít nhất một phép nhân.
- Mức 3: đường vòng `a → a·t → t` tốn đúng `x + y`; vậy đáp án là `x + y`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert (pair_cost(8, 6), pair_cost(6, 12), pair_cost(3, 1)) == (2, 1, INF)

# Bài 2
cnt = Counter([4, 6, 12])
assert sum(c * pair_cost(a, 12) for a, c in cnt.items()) == 2
assert sum(c * pair_cost(a, 4) for a, c in cnt.items()) == 3
assert sum(c * pair_cost(a, 6) for a, c in cnt.items()) == 3
assert Solution().minOperations([4, 6, 12]) == 2

# Bài 3
def pair_cost_todo_ans(a, t):
    if a == t:
        return 0
    if t == 1:
        return INF
    if a % t == 0 or t % a == 0:
        return 1
    return 2

for a in range(1, 40):
    for t in range(1, 40):
        assert pair_cost_todo_ans(a, t) == pair_cost(a, t)

# Bài 4
def my_solve_ans(nums):
    n = len(nums)
    cnt = Counter(nums)
    if set(cnt) == {1}:
        return 0
    best = n
    for t in cnt:
        if t == 1:
            continue
        best = min(best, sum(c * pair_cost_todo_ans(a, t) for a, c in cnt.items()))
    return best

rng = random.Random(41)
for _ in range(300):
    nums = [rng.randint(1, 60) for _ in range(rng.randint(1, 15))]
    assert my_solve_ans(nums) == Solution().minOperations(nums)
print("Bài 1–4 OK")

Bài 1–4 OK


In [28]:
# Bài 5: chi phí cặp có trọng số (nhân x, chia y), a, t >= 2
def pair_cost_xy_ans(a, t, x, y):
    if a == t:
        return 0
    if t % a == 0:
        return x                      # chỉ nhân
    if a % t == 0:
        return y                      # chỉ chia
    return x + y                      # a -> a*t (nhân) -> t (chia)

import heapq

def dijkstra_weighted(a, cap, x, y):
    dist = {a: 0}
    pq = [(0, a)]
    while pq:
        d, v = heapq.heappop(pq)
        if d > dist[v]:
            continue
        nbrs = [(m, x) for m in range(2 * v, cap + 1, v)]
        nbrs += [(v // k, y) for k in range(2, v) if v % k == 0]
        for u, w in nbrs:
            if d + w < dist.get(u, INF):
                dist[u] = d + w
                heapq.heappush(pq, (d + w, u))
    return dist

rng = random.Random(7)
for _ in range(60):
    a, t = rng.randint(2, 12), rng.randint(2, 12)
    x, y = rng.randint(1, 5), rng.randint(1, 5)
    got = dijkstra_weighted(a, 300, x, y).get(t, INF)
    assert got == pair_cost_xy_ans(a, t, x, y), (a, t, x, y, got)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "chi phí cặp ∈ {0,1,2}" + đếm ước/bội:**

```python
cnt = Counter(nums)
if set(cnt) == {1}: return 0
best = n                                     # đích là bội chung rất lớn
for m in cnt:                                # liệt kê ước của m bằng phân tích thừa số
    for d in divisors(m):
        if d != m and d in cnt:
            A[m] += cnt[d]                   # d | m, d < m
            B[d] += cnt[m]                   # m là bội của d
for t, c in cnt.items():
    if t == 1: continue                      # không ai về được 1
    best = min(best, 2*n - 2*c - A[t] - B[t])
return best
```

**Checklist trước khi submit:**

- [ ] Chi phí cặp: `0` nếu bằng, `1` nếu chia hết một chiều, `2` nếu không, `INF` nếu đích là `1`.
- [ ] Toàn số `1` → `0`; bỏ đích `t = 1` trong vòng lặp.
- [ ] `best` khởi tạo bằng `n` (đích bội chung lớn).
- [ ] `A` chỉ gồm ước **nhỏ hơn hẳn**, `B` chỉ gồm bội **lớn hơn hẳn**.
- [ ] Phân tích thừa số dừng khi `p*p > x`, phần còn lại `> 1` là số nguyên tố.
- [ ] Đã đối chiếu BFS trên giá trị nhỏ; đã đối chiếu lại tên phương thức nếu có tài khoản Premium.